# Chapitre 11 - Cours 2 : ajuster un réseau de neurones (rétropropagation, weight decay)

Ce notebook couvre les sections 11.4 et 11.5 du livre :

1. les critères d'ajustement (erreur quadratique, entropie croisée, éq. (11.9)–(11.10)) ;
2. la **rétropropagation** du gradient, dérivée pas à pas (éq. (11.11)–(11.15)), implémentée en
   numpy puis **vérifiée** par différences finies ;
3. la descente de gradient (éq. (11.13)) : par lots, en ligne, notion d'époque ;
4. les questions pratiques de la section 11.5 : valeurs initiales, sur-ajustement et arrêt
   précoce, **weight decay** (éq. (11.16)–(11.17)), mise à l'échelle des entrées, minima
   multiples ;
5. l'exemple du **mélange** du chapitre 2 (figures 11.4 et 11.5).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch11_reseaux import (MLP, backprop, forward, gradient_check, init_params,
                              numerical_gradient, one_hot, pack, sigmoid, unpack)

## 1. Le critère à minimiser

Rappelons le modèle (11.5) : pour une observation $x_i$,

$$z_{mi} = \sigma(\alpha_{0m} + \alpha_m^T x_i), \qquad T_{ki} = \beta_{0k} + \beta_k^T z_i, \qquad f_k(x_i) = g_k(T_i),$$

où $z_i = (z_{1i}, \dots, z_{Mi})$. L'ensemble $\theta$ des poids (on dit souvent *poids* pour
*paramètres*) comporte $M(p+1) + K(M+1)$ nombres (éq. (11.8)).

* En **régression**, on minimise la somme des carrés des erreurs

$$R(\theta) = \sum_{k=1}^K \sum_{i=1}^N \big(y_{ik} - f_k(x_i)\big)^2 \qquad \text{(éq. (11.9))}.$$

* En **classification**, avec $y_{ik} = 1$ si l'observation $i$ est de classe $k$ et 0 sinon, on
  utilise soit l'erreur quadratique, soit l'**entropie croisée** (ou déviance)

$$R(\theta) = -\sum_{i=1}^N \sum_{k=1}^K y_{ik} \log f_k(x_i) \qquad \text{(éq. (11.10))},$$

  et le classifieur est $G(x) = \arg\max_k f_k(x)$.

> **Remarque.** Avec la sortie softmax et l'entropie croisée, $R(\theta)$ est exactement
> l'opposé de la log-vraisemblance d'un modèle logistique multinomial **en les unités
> cachées** $z_m$ : ajuster le réseau, c'est faire du maximum de vraisemblance.
>
> En général on ne veut **pas** le minimum global de $R$, qui sur-ajuste presque toujours :
> il faut régulariser, soit par une pénalité, soit en arrêtant l'optimisation tôt (section 4).

## 2. La rétropropagation pas à pas

La méthode générique pour minimiser $R(\theta)$ est la **descente de gradient**. Il faut donc
calculer les dérivées partielles de $R$ par rapport à **chaque** poids. Grâce à la structure en
couches du modèle (composition de fonctions), la **règle de dérivation en chaîne** permet de les
obtenir avec deux balayages du réseau : c'est la **rétropropagation** (*back-propagation*).

Nous traitons l'erreur quadratique (l'entropie croisée fait l'objet de l'exercice 11.3).
Comme $R$ est une somme sur les observations, il suffit de dériver chaque terme :

$$R(\theta) = \sum_{i=1}^N R_i, \qquad R_i = \sum_{k=1}^K \big(y_{ik} - f_k(x_i)\big)^2 \qquad \text{(éq. (11.11))}.$$

### Rappel : la règle de dérivation en chaîne

Si une quantité $R$ dépend d'une variable $u$ **seulement à travers** des quantités
intermédiaires $v_1, \dots, v_K$ (c'est-à-dire $R = \Phi(v_1(u), \dots, v_K(u))$), alors

$$\frac{\partial R}{\partial u} = \sum_{k=1}^K \frac{\partial R}{\partial v_k} \, \frac{\partial v_k}{\partial u}.$$

Le principe de la rétropropagation est de **mémoriser** les dérivées $\partial R / \partial v_k$
par rapport aux quantités intermédiaires, puis de les réutiliser pour toutes les variables
situées « en amont ».

Nous notons $A_{mi} = \alpha_{0m} + \alpha_m^T x_i$ l'entrée de l'unité cachée $m$ (de sorte que
$z_{mi} = \sigma(A_{mi})$), et nous supposons que chaque $g_k$ ne dépend que de $T_k$ (identité ou
sigmoïde : $f_k = g_k(T_k)$). Nous posons aussi $z_{0i} = 1$ et $x_{i0} = 1$ pour traiter les
constantes comme des poids ordinaires.

### Étape 1 : dérivées par rapport aux poids de sortie $\beta_{km}$

Le poids $\beta_{km}$ n'intervient que dans $T_{ki} = \beta_{0k} + \sum_{m'} \beta_{km'} z_{m'i}$, et
$\partial T_{ki} / \partial \beta_{km} = z_{mi}$. Par la règle de la chaîne,

$$\frac{\partial R_i}{\partial \beta_{km}} = \frac{\partial R_i}{\partial T_{ki}} \cdot z_{mi}.$$

Or $T_{ki}$ n'intervient dans $R_i$ qu'à travers le terme $(y_{ik} - g_k(T_{ki}))^2$, d'où

$$\frac{\partial R_i}{\partial T_{ki}} = -2\big(y_{ik} - f_k(x_i)\big)\, g_k'(T_{ki}).$$

Nous obtenons la première formule de (11.12) :

$$\frac{\partial R_i}{\partial \beta_{km}} = -2\big(y_{ik} - f_k(x_i)\big)\, g_k'(\beta_k^T z_i)\, z_{mi}.$$

### Étape 2 : dérivées par rapport aux poids cachés $\alpha_{m\ell}$

Le poids $\alpha_{m\ell}$ n'intervient que dans $A_{mi}$ (avec
$\partial A_{mi}/\partial \alpha_{m\ell} = x_{i\ell}$), donc seulement dans $z_{mi}$. Mais $z_{mi}$
intervient dans **toutes** les sorties $T_{1i}, \dots, T_{Ki}$, avec
$\partial T_{ki}/\partial z_{mi} = \beta_{km}$. La règle de la chaîne donne donc une **somme sur $k$** :

$$
\begin{aligned}
\frac{\partial R_i}{\partial \alpha_{m\ell}}
&= \sum_{k=1}^K \frac{\partial R_i}{\partial T_{ki}} \cdot \frac{\partial T_{ki}}{\partial z_{mi}} \cdot \frac{\partial z_{mi}}{\partial A_{mi}} \cdot \frac{\partial A_{mi}}{\partial \alpha_{m\ell}} \\
&= -\sum_{k=1}^K 2\big(y_{ik} - f_k(x_i)\big)\, g_k'(\beta_k^T z_i)\, \beta_{km}\, \sigma'(\alpha_m^T x_i)\, x_{i\ell},
\end{aligned}
$$

ce qui est la seconde formule de (11.12) (le livre écrit $\alpha_m^T x_i$ pour $A_{mi}$, la constante
étant incluse).

### Étape 3 : les « erreurs » $\delta$ et $s$

On remarque que les deux formules partagent les mêmes facteurs. Posons

$$\delta_{ki} = \frac{\partial R_i}{\partial T_{ki}} = -2\big(y_{ik} - f_k(x_i)\big)\, g_k'(\beta_k^Tz_i), \qquad
s_{mi} = \frac{\partial R_i}{\partial A_{mi}}.$$

Alors (11.12) s'écrit simplement

$$\frac{\partial R_i}{\partial \beta_{km}} = \delta_{ki}\, z_{mi}, \qquad \frac{\partial R_i}{\partial \alpha_{m\ell}} = s_{mi}\, x_{i\ell}
\qquad \text{(éq. (11.14))},$$

et, en comparant avec l'étape 2, les erreurs cachées s'obtiennent à partir des erreurs de sortie :

$$s_{mi} = \sigma'(\alpha_m^T x_i) \sum_{k=1}^K \beta_{km}\, \delta_{ki} \qquad \text{(éq. (11.15))}.$$

C'est l'**équation de rétropropagation** : l'erreur d'une unité cachée est la somme des erreurs
des sorties auxquelles elle est reliée, pondérées par les poids de ces liaisons, puis
multipliée par la pente locale $\sigma'$. Rappelons que $\sigma' = \sigma(1-\sigma)$, donc
$\sigma'(A_{mi}) = z_{mi}(1 - z_{mi})$ : aucune exponentielle supplémentaire n'est nécessaire.

### L'algorithme en deux passes

1. **Passe avant** : avec les poids courants, calculer $A_{mi}$, $z_{mi}$, $T_{ki}$ puis $f_k(x_i)$.
2. **Passe arrière** : calculer les erreurs de sortie $\delta_{ki}$, puis les **rétropropager**
   par (11.15) pour obtenir les $s_{mi}$.
3. Assembler les gradients par (11.14) en sommant sur $i$.

> **Intuition.** Chaque unité n'a besoin que d'informations **locales** : ses entrées dans la
> passe avant, et les erreurs des unités auxquelles elle envoie son signal dans la passe arrière.
> Le coût total est du même ordre que celui d'une seule évaluation du réseau, soit
> $O(N \times \text{nombre de poids})$, au lieu de $O(N \times \text{nombre de poids}^2)$ pour des
> différences finies.

### Forme matricielle

Rangeons les quantités dans des matrices : $\mathbf X$ ($N \times p$), $\mathbf Z$ ($N\times M$, de
terme $z_{mi}$), $\boldsymbol\Delta$ ($N\times K$, de terme $\delta_{ki}$), $\mathbf S$ ($N\times M$),
$\mathbf B$ ($K\times M$, poids $\beta_{km}$ sans les constantes). Alors, en notant $\odot$ le produit
terme à terme et $\mathbf 1$ le vecteur de uns :

$$\mathbf S = (\boldsymbol\Delta\,\mathbf B) \odot \mathbf Z \odot (1 - \mathbf Z), \qquad
\frac{\partial R}{\partial \mathbf B} = \boldsymbol\Delta^T \mathbf Z,\quad \frac{\partial R}{\partial \beta_{0}} = \boldsymbol\Delta^T \mathbf 1, \qquad
\frac{\partial R}{\partial \boldsymbol\alpha} = \mathbf S^T \mathbf X,\quad \frac{\partial R}{\partial \alpha_{0}} = \mathbf S^T\mathbf 1.$$

Voici une implémentation directe (régression, sortie identité, donc $g_k' = 1$).

In [2]:
def passe_avant(alpha, beta, X):
    """alpha : (M, p+1), beta : (K, M+1). Renvoie Z (N, M) et F (N, K)."""
    A = alpha[:, 0] + X @ alpha[:, 1:].T          # A_mi = alpha_0m + alpha_m^T x_i
    Z = 1 / (1 + np.exp(-A))                      # z_mi = sigma(A_mi)
    F = beta[:, 0] + Z @ beta[:, 1:].T            # f_k(x_i) = T_ki (sortie identité)
    return Z, F

In [ ]:
def retropropagation(alpha, beta, X, Y):
    """Erreur quadratique R et gradients (dR/dalpha, dR/dbeta)."""
    Z, F = passe_avant(alpha, beta, X)
    R = np.sum((Y - F) ** 2)
    Delta = -2 * (Y - F)                          # delta_ki (g' = 1)
    S = (Delta @ beta[:, 1:]) * Z * (1 - Z)       # s_mi, éq. (11.15)
    g_beta = np.column_stack([Delta.sum(axis=0), Delta.T @ Z])   # éq. (11.14)
    g_alpha = np.column_stack([S.sum(axis=0), S.T @ X])
    return R, g_alpha, g_beta